In [1]:
import os
import random

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

c:\Users\Lefteris Dragasakis\Documents\GitHub\Supervised-Learning-Experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

In [3]:
# Load data from CSV files
train_data = pd.read_csv("./train_data.csv")
test_data = pd.read_csv("./test_data.csv")

# map columns 'c' and 'mask' to lists of integers
to_list = lambda x: list(map(int, x.split(",")))
train_data["c"] = train_data["c"].apply(to_list)
train_data["mask"] = train_data["mask"].apply(to_list)
test_data["c"] = test_data["c"].apply(to_list)

print(f"Loaded {len(train_data)} training examples and {len(test_data)} test examples.")

Loaded 5000 training examples and 500 test examples.


In [4]:
# Load model and tokenizer
model_path = "./pythia-14m"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).to(device)
model.eval()
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(model)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 4491.89it/s]

Vocabulary size: 50254
GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 128)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-5): 6 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (query_key_value): Linear(in_features=128, out_features=384, bias=True)
          (dense): Linear(in_features=128, out_features=128, bias=True)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=128, out_features=512, bias=True)
          (dense_4h_to_h): Linear(in_features=512, out_features=128, bias=True)
          (act): GELUActivation()
        )
      )
    )
    (final_layer_norm): LayerNorm((128,), ep

In [5]:
# View the first example in human readable format.
example = train_data.iloc[0]
a = np.array(example['c'])[np.array(example['mask']) == 0]
b = np.array(example['c'])[np.array(example['mask']) == 1]

print(f"id: {example['id']}, a length: {len(a)}, b length: {len(b)}, c length: {len(example['c'])}")
print(f"Sentence A (first 30 tokens): {tokenizer.decode(a[:30])}...")
print(f"Sentence B (first 30 tokens): {tokenizer.decode(b[:30])}...")
print(f"Interleaved (first 30 tokens): {tokenizer.decode(example['c'][:30])}...")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPTNeoXTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


id: 0, a length: 131, b length: 180, c length: 311
Sentence A (first 30 tokens): the ancient spiritual system in oriental world stressed on simple and minimalist living so that there is less pressure on demand side and therefore, less propensity of...
Sentence B (first 30 tokens): however, we in the new millennium have to be more aware and better endowed to handle this all. we surely do not accept to stand in 'den...
Interleaved (first 30 tokens): however, we inthe the ancient new millennium spiritual have system in to ori be more aware andental world stressed better endowed on to simple handle this and...


In [9]:
# TODO: Predict the masks for the examples in test set

def predict_mask(example):
    # TODO: placeholder
    # noone in the whole competition didn't get more than the baseline on this one! I am curious finding the correct solution
    return [random.choice(range(3)) * len(example['c']) 

_IncompleteInputError: incomplete input (4126123268.py, line 6)

In [7]:
def evaluate(data, predict_fn):
    """Evaluate predict_fn on labeled data using symmetric mask accuracy."""
    total_acc = 0.0
    for example in data:
        true_mask = np.array(example['mask'])
        pred_mask = np.array(predict_fn(example))
        if len(pred_mask) != len(true_mask):
            continue
        acc = np.mean(pred_mask == true_mask)
        total_acc += max(acc, 1.0 - acc)
    return total_acc / len(data) if data else None

# Example: evaluate on training data
# accuracy = evaluate(train_data.to_dict('records'), predict_mask)
# print(f"Train accuracy: {accuracy:.4f}")


In [8]:
# create a submission csv file

with open('submission.csv', 'w') as f:
    f.write('subtaskID,datapointID,answer\n')
    for _, example in test_data.iterrows():
        mask = predict_mask(example)
        answer = '"' + ','.join(str(m) for m in mask) + '"'
        f.write(f'"1","{example["id"]}",{answer}\n')